# LeetCode #89 — Gray Code: phản chiếu (reflect) và công thức `i ^ (i >> 1)`

**Mức độ:** Medium · **Chủ đề:** Math, Backtracking, Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def grayCode(self, n: int) -> List[int]:
```

Mục tiêu của notebook:

1. Hiểu **mã Gray**: hai số liên tiếp khác nhau đúng **một bit**.
2. Cách 1 — **phản chiếu (reflect-and-prefix)**: dựng mã `n` bit từ mã `n − 1` bit.
3. Cách 2 — công thức đóng **`g(i) = i ^ (i >> 1)`** và vì sao nó đúng.
4. Cách 3 — backtracking (tìm chu trình Hamilton trên hypercube) để thấy vì sao hai cách trên hay hơn.
5. Các thao tác bit: `^`, `>>`, `<<`, `bin(x).count("1")`.

# 1. Bài toán là gì?

**Dãy mã Gray `n` bit** là dãy `2^n` số nguyên thỏa:

- Mỗi số nằm trong `[0, 2^n − 1]`.
- Số **đầu tiên** là `0`.
- Mỗi số xuất hiện **không quá một lần**.
- Biểu diễn nhị phân của hai số **liền kề** khác nhau **đúng một bit**.
- Số **đầu** và số **cuối** cũng khác nhau đúng một bit (dãy là một **vòng**).

Cho `n`, trả về **một** dãy mã Gray `n` bit bất kỳ.

**Constraints (theo đề):** `1 <= n <= 16`

---

## Ví dụ

```text
n = 2  ->  [0, 1, 3, 2]      00, 01, 11, 10
n = 1  ->  [0, 1]
```

Một đáp án khác cũng đúng cho `n = 2`: `[0, 2, 3, 1]` (`00, 10, 11, 01`).

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Biểu diễn nhị phân và các phép toán bit
2. Đệ quy (recursion)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Biểu diễn nhị phân và các phép toán bit

### Số nguyên dưới dạng bit

Máy tính lưu số nguyên ở hệ **nhị phân**: mỗi chữ số (bit) là 0 hoặc 1, bit thứ `k` (đếm từ 0, bên phải) có giá trị `2^k`.

```text
13 = 8 + 4 + 1 = 1·2³ + 1·2² + 0·2¹ + 1·2⁰ = 0b1101
bit:              3     2      1      0
```

In [1]:
x_bit = 13
print("bin(13) =", bin(x_bit), "| format 8 bit:", format(x_bit, "08b"), "| int('1101', 2) =", int("1101", 2))
print("bit thứ k của 13:", [(x_bit >> k) & 1 for k in range(4)], "(từ bit 0)")

bin(13) = 0b1101 | format 8 bit: 00001101 | int('1101', 2) = 13
bit thứ k của 13: [1, 0, 1, 1] (từ bit 0)


### Sáu phép toán bit

| Phép | Ký hiệu | Quy tắc từng bit | Ví dụ `12 = 1100`, `10 = 1010` |
|---|---|---|---|
| AND | `a & b` | 1 nếu **cả hai** là 1 | `1000` = 8 |
| OR | `a \| b` | 1 nếu **ít nhất một** là 1 | `1110` = 14 |
| XOR | `a ^ b` | 1 nếu **khác nhau** | `0110` = 6 |
| NOT | `~a` | đảo mọi bit | `~12 = -13` (Python) |
| dịch trái | `a << k` | thêm `k` số 0 bên phải = nhân `2^k` | `12 << 1 = 24` |
| dịch phải | `a >> k` | bỏ `k` bit phải = chia nguyên `2^k` | `12 >> 2 = 3` |

In [2]:
a_b, b_b = 12, 10
for ten, v in [("a & b", a_b & b_b), ("a | b", a_b | b_b), ("a ^ b", a_b ^ b_b), ("a << 1", a_b << 1), ("a >> 2", a_b >> 2), ("~a", ~a_b)]:
    print(f"{ten:<7} = {v:>4}   bin: {bin(v)}")

a & b   =    8   bin: 0b1000
a | b   =   14   bin: 0b1110
a ^ b   =    6   bin: 0b110
a << 1  =   24   bin: 0b11000
a >> 2  =    3   bin: 0b11
~a      =  -13   bin: -0b1101


### Các mẹo hay dùng

| Mục đích | Biểu thức |
|---|---|
| Kiểm tra bit `k` | `(x >> k) & 1` |
| Bật / tắt / lật bit `k` | `x \| (1 << k)`, `x & ~(1 << k)`, `x ^ (1 << k)` |
| Số chẵn? | `x & 1 == 0` |
| Bỏ bit 1 thấp nhất | `x & (x - 1)` |
| Lấy bit 1 thấp nhất | `x & -x` |
| Là lũy thừa của 2? | `x > 0 and x & (x - 1) == 0` |

**Tính chất XOR** (cực kỳ hay dùng): `x ^ x = 0`, `x ^ 0 = x`, giao hoán và kết hợp → XOR cả dãy thì các số xuất hiện **chẵn** lần triệt tiêu nhau.

In [3]:
x_m = 0b10110
print("bỏ bit 1 thấp nhất:", format(x_m, "05b"), "→", format(x_m & (x_m - 1), "05b"))
print("lấy bit 1 thấp nhất:", format(x_m & -x_m, "05b"))
print("lũy thừa của 2:", [v for v in range(1, 40) if v & (v - 1) == 0])
print("đếm bit 1 của 255:", bin(255).count("1"))
xor_all = 0
for v in [4, 1, 2, 1, 2]:
    xor_all ^= v
print("XOR [4,1,2,1,2] =", xor_all, "(số xuất hiện lẻ lần)")

bỏ bit 1 thấp nhất: 10110 → 10100
lấy bit 1 thấp nhất: 00010
lũy thừa của 2: [1, 2, 4, 8, 16, 32]
đếm bit 1 của 255: 8
XOR [4,1,2,1,2] = 4 (số xuất hiện lẻ lần)


### Dùng bit làm tập hợp (bitmask)

Với `n` phần tử, một số `mask` từ `0` tới `2^n − 1` biểu diễn một **tập con**: bit `i` bằng 1 nghĩa là lấy phần tử `i`. Duyệt mọi `mask` = duyệt mọi tập con.

In [4]:
items = ["a", "b", "c"]
for mask in range(1 << len(items)):
    tap = [items[i] for i in range(len(items)) if mask >> i & 1]
    print(format(mask, "03b"), tap)

000 []
001 ['a']
010 ['b']
011 ['a', 'b']
100 ['c']
101 ['a', 'c']
110 ['b', 'c']
111 ['a', 'b', 'c']


### Số âm và giới hạn 32 bit

Ngôn ngữ như C/Java lưu `int` bằng 32 bit, số âm dạng **bù hai** (two's complement): `-x = ~x + 1`. Python có số nguyên **không giới hạn**, nên muốn mô phỏng 32 bit phải **che** bằng `& 0xFFFFFFFF` rồi đổi lại thành số có dấu.

In [5]:
def thanh_32bit(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >> 31 else x       # bit 31 là bit dấu

print("-1 ở 32 bit:", format(-1 & 0xFFFFFFFF, "032b"))
print("2^31 tràn thành:", thanh_32bit(2 ** 31), "| (2^31 - 1) + 1 =", thanh_32bit(2 ** 31 - 1 + 1))

-1 ở 32 bit: 11111111111111111111111111111111
2^31 tràn thành: -2147483648 | (2^31 - 1) + 1 = -2147483648


Lỗi hay gặp: nhầm `^` là lũy thừa (lũy thừa là `**`); quên ngoặc vì `&`, `|`, `^` có độ ưu tiên **thấp hơn** `==` — viết `(x & 1) == 0`; `~x` trong Python là `-x - 1`, không phải "đảo 32 bit".

## 2.2 Đệ quy (recursion)

### Đệ quy là gì?

Một hàm **đệ quy** là hàm tự gọi lại chính nó trên một bài toán **nhỏ hơn** cùng dạng. Mỗi hàm đệ quy có hai phần:

1. **Trường hợp cơ sở (base case)**: bài toán đủ nhỏ để trả lời ngay, không gọi tiếp.
2. **Bước đệ quy**: chia nhỏ, gọi lại, rồi dùng kết quả để trả lời bài lớn.

Ví dụ `n! = n × (n−1)!`, với `0! = 1`.

In [6]:
def giai_thua(n):
    if n == 0:                  # trường hợp cơ sở
        return 1
    return n * giai_thua(n - 1) # bước đệ quy: bài nhỏ hơn
print([giai_thua(n) for n in range(7)])

[1, 1, 2, 6, 24, 120, 720]


### Ngăn xếp lời gọi (call stack)

Mỗi lần gọi hàm, máy tạo một **khung** (frame) chứa biến cục bộ và đặt lên **ngăn xếp lời gọi**. Hàm con chạy xong thì khung của nó bị bỏ, kết quả trả về cho khung bên dưới. Ta in thụt lề theo độ sâu để thấy rõ:

In [7]:
def giai_thua_vet(n, sau=0):
    le = "    " * sau
    print(f"{le}gọi giai_thua({n})")
    if n == 0:
        print(f"{le}→ trả 1 (cơ sở)")
        return 1
    kq = n * giai_thua_vet(n - 1, sau + 1)
    print(f"{le}→ trả {n} × ... = {kq}")
    return kq
_ = giai_thua_vet(3)

gọi giai_thua(3)
    gọi giai_thua(2)
        gọi giai_thua(1)
            gọi giai_thua(0)
            → trả 1 (cơ sở)
        → trả 1 × ... = 1
    → trả 2 × ... = 2
→ trả 3 × ... = 6


### Cách viết một hàm đệ quy đúng

1. **Định nghĩa rõ** hàm trả về gì cho đầu vào nào ("`f(node)` = chiều cao cây gốc `node`").
2. **Tin** rằng lời gọi trên bài nhỏ hơn đã trả đúng (giả thiết quy nạp).
3. Ghép kết quả các bài nhỏ thành kết quả bài lớn.
4. Kiểm tra trường hợp cơ sở được chạm tới với **mọi** đầu vào (mỗi lần gọi phải nhỏ đi).

Ví dụ: tổng các chữ số, lũy thừa nhanh.

In [8]:
def tong_chu_so(n):
    if n < 10:
        return n
    return n % 10 + tong_chu_so(n // 10)

def luy_thua(x, k):              # x^k, chia đôi số mũ
    if k == 0:
        return 1
    nua = luy_thua(x, k // 2)
    return nua * nua * (x if k % 2 else 1)

print("tong_chu_so(98765) =", tong_chu_so(98765), "| luy_thua(3, 13) =", luy_thua(3, 13), "=", 3 ** 13)

tong_chu_so(98765) = 35 | luy_thua(3, 13) = 1594323 = 1594323


### Ghi nhớ kết quả (memoization)

Một số hàm đệ quy gọi **lặp lại** cùng một bài con rất nhiều lần, ví dụ Fibonacci `F(n) = F(n−1) + F(n−2)`: `F(5)` gọi `F(3)` hai lần, `F(2)` ba lần… → `O(2^n)`.

**Memoization**: lưu kết quả đã tính vào dict (hoặc dùng `@lru_cache`), lần sau gặp lại thì trả ngay → mỗi bài con tính đúng một lần.

In [9]:
from functools import lru_cache

dem_goi = 0
def fib_cham(n):
    global dem_goi
    dem_goi += 1
    return n if n < 2 else fib_cham(n - 1) + fib_cham(n - 2)

memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

@lru_cache(maxsize=None)
def fib_cache(n):
    return n if n < 2 else fib_cache(n - 1) + fib_cache(n - 2)

print("fib_cham(25) =", fib_cham(25), "với", dem_goi, "lần gọi")
print("fib_memo(25) =", fib_memo(25), "| bảng nhớ có", len(memo), "mục")
print("fib_cache(90) =", fib_cache(90))

fib_cham(25) = 75025 với 242785 lần gọi
fib_memo(25) = 75025 | bảng nhớ có 24 mục
fib_cache(90) = 2880067194370816120


### Lỗi hay gặp

- **Thiếu / sai trường hợp cơ sở** → đệ quy vô hạn → `RecursionError` (Python mặc định giới hạn ~1000 tầng).
- Bài con **không nhỏ đi** (ví dụ gọi `f(n)` trong `f(n)`).
- Đệ quy quá sâu với dữ liệu lớn (danh sách 10^5 nút) → đổi sang vòng lặp hoặc tăng `sys.setrecursionlimit` (có rủi ro).
- Dùng list/biến dùng chung giữa các lời gọi mà quên "trả lại" trạng thái (xem backtracking).

In [10]:
def thieu_co_so(n):
    return thieu_co_so(n - 1)
try:
    thieu_co_so(5)
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")

RecursionError: maximum recursion depth exceeded ...


## 2.3 Áp dụng vào bài này

Mã Gray: hai số liền kề khác nhau đúng **một bit**. Cách **đệ quy** "phản chiếu": dãy cho `n` bit = dãy `n−1` bit, nối với chính nó **đảo ngược** và bật bit `n−1`. Công thức trực tiếp: `g(i) = i ^ (i >> 1)` — dùng phép **XOR** và **dịch bit**.

# 3. Nền tảng: các phép toán bit

| Phép | Ý nghĩa | Ví dụ |
|---|---|---|
| `a ^ b` | XOR từng bit (khác nhau → 1) | `0b110 ^ 0b011 = 0b101` |
| `a >> k` | dịch phải `k` bit (chia nguyên cho `2^k`) | `0b110 >> 1 = 0b11` |
| `1 << k` | `2^k` — số chỉ có bit thứ `k` bật | `1 << 3 = 8` |
| `bin(x).count("1")` | số bit 1 (popcount) | `bin(5).count("1") = 2` |

Hai số `a`, `b` khác nhau **đúng một bit** ⇔ `a ^ b` là một lũy thừa của 2 ⇔ `bin(a ^ b).count("1") == 1`.

In [11]:
from typing import List

def one_bit_diff(a: int, b: int) -> bool:
    return bin(a ^ b).count("1") == 1

print(f"{0b110 ^ 0b011:03b}", 0b110 >> 1, 1 << 3, bin(5).count("1"))
print(one_bit_diff(1, 3), one_bit_diff(1, 2))

101 3 8 2
True False


Hàm kiểm tra một dãy có phải mã Gray `n` bit hợp lệ:

In [12]:
def is_gray(seq: List[int], n: int) -> bool:
    if len(seq) != 1 << n or seq[0] != 0:
        return False
    if sorted(seq) != list(range(1 << n)):
        return False
    return all(one_bit_diff(seq[i], seq[(i + 1) % len(seq)]) for i in range(len(seq)))

print(is_gray([0, 1, 3, 2], 2), is_gray([0, 2, 3, 1], 2), is_gray([0, 1, 2, 3], 2))

True True False


# 4. Cách 1: Phản chiếu (reflect-and-prefix)

Giả sử đã có dãy Gray `G(n−1)` gồm `2^(n−1)` số. Dựng `G(n)`:

1. Giữ nguyên `G(n−1)` (bit cao nhất = 0).
2. **Đảo ngược** `G(n−1)` rồi bật bit thứ `n−1` (cộng `2^(n−1)`) cho từng số.
3. Nối hai phần.

```text
G(1) = 0, 1
G(2) = 00, 01 | 11, 10            (phần sau: đảo ngược 0,1 rồi thêm bit 1 phía trước)
G(3) = 000, 001, 011, 010 | 110, 111, 101, 100
```

In [13]:
def gray_reflect(n: int) -> List[int]:
    result = [0]
    for bit in range(n):
        result += [x | (1 << bit) for x in reversed(result)]
    return result

for n in range(1, 4):
    g = gray_reflect(n)
    print(n, g, [f"{x:0{n}b}" for x in g])

1 [0, 1] ['0', '1']
2 [0, 1, 3, 2] ['00', '01', '11', '10']
3 [0, 1, 3, 2, 6, 7, 5, 4] ['000', '001', '011', '010', '110', '111', '101', '100']


**Vì sao đúng?**

- Bên trong mỗi nửa: hai số liền kề vẫn khác một bit (nửa sau là nửa đầu đảo ngược, thêm cùng một bit).
- **Chỗ nối:** số cuối của nửa đầu và số đầu của nửa sau là **cùng một** số của `G(n−1)`, chỉ khác bit thứ `n−1` → khác đúng một bit.
- **Vòng:** số đầu `0` và số cuối `0 | (1 << (n−1))` (đầu của `G(n−1)`, sau khi đảo ngược thì nằm cuối) khác đúng bit `n−1`.

# 5. Cách 2: Công thức `g(i) = i ^ (i >> 1)`

Số Gray thứ `i` (theo thứ tự phản chiếu ở trên) bằng `i XOR (i dịch phải 1)`.

In [14]:
def gray_formula(n: int) -> List[int]:
    return [i ^ (i >> 1) for i in range(1 << n)]

print(gray_formula(3))
print("Giống cách phản chiếu?", all(gray_formula(n) == gray_reflect(n) for n in range(1, 12)))

[0, 1, 3, 2, 6, 7, 5, 4]
Giống cách phản chiếu? True


## Vì sao `i` và `i + 1` cho mã Gray khác đúng một bit?

Khi tăng `i` lên 1, các bit thay đổi là: một dãy bit `1` ở cuối thành `0`, và bit `0` ngay trước nó thành `1`. Ví dụ `i = 0b10111 → i + 1 = 0b11000`: đổi `k + 1 = 4` bit cuối.

`g(i) = i ^ (i >> 1)`: bit thứ `j` của `g` là `bit_j(i) XOR bit_(j+1)(i)` — tức là "bit `j` có khác bit ngay trên nó không".

Gọi `k` = số bit 1 ở cuối của `i`. Khi tăng lên `i + 1`:

| Bit | trong `i` | trong `i + 1` |
|---|---|---|
| `0 .. k−1` | toàn `1` | toàn `0` |
| `k` | `0` | `1` |
| `> k` | không đổi | không đổi |

- Với `j < k − 1`: hai bit `j`, `j+1` **bằng nhau** ở cả `i` (cùng là 1) và `i + 1` (cùng là 0) → `g_j = 0` ở cả hai.
- Với `j = k − 1`: ở `i` là `1, 0`; ở `i + 1` là `0, 1` → `g_j = 1` ở cả hai.
- Với `j = k`: bit `k` đổi, bit `k + 1` không đổi → `g_k` **đổi**.
- Với `j > k`: không đổi.

Vậy `g(i)` và `g(i + 1)` khác **đúng bit `k`** = số bit 0 ở cuối của `i + 1`.

Kiểm chứng bằng code:

In [15]:
def trace_formula(n: int):
    print(f"{'i':>3} {'i (bin)':>8} {'i>>1':>6} {'g = i^(i>>1)':>13}  bit đổi so với g trước")
    prev = None
    for i in range(1 << n):
        g = i ^ (i >> 1)
        changed = "" if prev is None else f"bit {(prev ^ g).bit_length() - 1}"
        print(f"{i:>3} {i:>8b} {i >> 1:>6b} {g:>13b}  {changed}")
        prev = g

trace_formula(3)

  i  i (bin)   i>>1  g = i^(i>>1)  bit đổi so với g trước
  0        0      0             0  
  1        1      0             1  bit 0
  2       10      1            11  bit 1
  3       11      1            10  bit 0
  4      100     10           110  bit 2
  5      101     10           111  bit 0
  6      110     11           101  bit 1
  7      111     11           100  bit 0


Cột cuối cho thấy bit bị đổi tại bước `i` chính là **số bit 0 ở cuối của `i`** (bit thấp nhất bật trong `i`) — đúng như lập luận.

# 6. Cách 3: Backtracking (để so sánh)

Xem mỗi số `0 .. 2^n − 1` là một đỉnh của **hypercube** `n` chiều, hai đỉnh nối nhau nếu khác một bit. Mã Gray = **chu trình Hamilton** đi qua mọi đỉnh. DFS + quay lui: từ số hiện tại, thử lật từng bit, nếu số mới chưa dùng thì đi tiếp.

In [16]:
def gray_backtrack(n: int) -> List[int]:
    total = 1 << n
    path, used = [0], {0}
    def dfs():
        if len(path) == total:
            return one_bit_diff(path[-1], path[0])
        for b in range(n):
            nxt = path[-1] ^ (1 << b)
            if nxt not in used:
                used.add(nxt); path.append(nxt)
                if dfs():
                    return True
                used.remove(nxt); path.pop()
        return False
    dfs()
    return path

print(gray_backtrack(3), is_gray(gray_backtrack(3), 3))

[0, 1, 3, 2, 6, 7, 5, 4] True


Với hypercube, lật bit thấp nhất có thể **tình cờ** cho ra đúng dãy phản chiếu mà không cần quay lui. Nhưng backtracking dùng đệ quy sâu `2^n` (với `n = 16` là 65 536 tầng — vượt giới hạn đệ quy mặc định của Python) và về lý thuyết là hàm mũ. Hai cách đầu chỉ `O(2^n)`, không đệ quy.

# 7. Phiên bản đúng format LeetCode

In [17]:
class Solution:
    def grayCode(self, n: int) -> List[int]:
        return [i ^ (i >> 1) for i in range(1 << n)]

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<14} PASS  {actual if len(str(actual)) < 60 else str(actual)[:57] + '...'}")


sol = Solution()
check("Ví dụ 1", sol.grayCode(2), [0, 1, 3, 2])
check("Ví dụ 2", sol.grayCode(1), [0, 1])
check("n = 3", sol.grayCode(3), [0, 1, 3, 2, 6, 7, 5, 4])

for n in range(1, 17):
    assert is_gray(sol.grayCode(n), n)
    assert is_gray(gray_reflect(n), n)
for n in range(1, 8):
    assert is_gray(gray_backtrack(n), n)
print("Mọi n từ 1 tới 16: dãy hợp lệ (backtracking kiểm tới n = 7)")

Ví dụ 1        PASS  [0, 1, 3, 2]
Ví dụ 2        PASS  [0, 1]
n = 3          PASS  [0, 1, 3, 2, 6, 7, 5, 4]


Mọi n từ 1 tới 16: dãy hợp lệ (backtracking kiểm tới n = 7)


# 8. Vì sao đúng? Độ phức tạp?

Đã chứng minh ở mục 4 (phản chiếu, quy nạp theo `n`) và mục 5 (công thức). Ngoài ra `g` là **song ánh** trên `[0, 2^n)`: từ `g` có thể khôi phục `i` (Bài 4) → mỗi số xuất hiện đúng một lần.

```text
Time:  O(2^n)
Space: O(2^n) cho output (không tính output: O(1))
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Không đảo ngược nửa đầu khi phản chiếu

In [19]:
def wrong_no_reverse(n: int) -> List[int]:
    result = [0]
    for bit in range(n):
        result += [x | (1 << bit) for x in result]       # SAI: thiếu reversed
    return result

w = wrong_no_reverse(2)
print("Sai :", w, [f"{x:02b}" for x in w], "| hợp lệ?", is_gray(w, 2))

Sai : [0, 1, 2, 3] ['00', '01', '10', '11'] | hợp lệ? False


Tại chỗ nối `01 → 10` khác **hai** bit.

## Lỗi 2: Dùng `i ^ (i << 1)` hoặc `i >> 1` một mình

In [20]:
w1 = [i ^ (i << 1) for i in range(4)]
w2 = [i >> 1 for i in range(4)]
print("i ^ (i << 1):", w1, "| hợp lệ?", is_gray(w1, 2))
print("i >> 1      :", w2, "| hợp lệ?", is_gray(w2, 2))

i ^ (i << 1): [0, 3, 6, 5] | hợp lệ? False
i >> 1      : [0, 0, 1, 1] | hợp lệ? False


## Lỗi 3: Viết `2 ^ n` để tính lũy thừa

Trong Python, `^` là **XOR**, không phải lũy thừa! `2 ^ 3 = 1`.

In [21]:
print("2 ^ 3 =", 2 ^ 3, "| 2 ** 3 =", 2 ** 3, "| 1 << 3 =", 1 << 3)

2 ^ 3 = 1 | 2 ** 3 = 8 | 1 << 3 = 8


## Lỗi 4: Quên điều kiện "vòng" (đầu và cuối khác một bit)

Một số dãy thỏa "liền kề khác một bit" nhưng số cuối và số đầu khác nhiều bit — đề yêu cầu cả điều kiện vòng. Hàm `is_gray` ở mục 3 kiểm tra cả cặp `(cuối, đầu)` bằng `(i + 1) % len`.

# 10. Mẹo nhận diện pattern

- **"Dãy liên tiếp khác một bit"** → mã Gray, `i ^ (i >> 1)`.
- **Dựng lời giải cỡ `n` từ cỡ `n − 1`** bằng cách phản chiếu / đối xứng — xuất hiện cả trong Tháp Hà Nội (số bước của Tháp Hà Nội liên hệ trực tiếp với mã Gray).
- Bài liên quan: **#1238 Circular Permutation in Binary Representation** (mã Gray bắt đầu từ `start`), **#1611 Minimum One Bit Operations to Make Integers Zero** (Hard, ngược của mã Gray), **#78 Subsets** (Bài 5 về thứ tự Gray).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Viết dãy mã Gray 4 bit bằng cách phản chiếu từ `G(3) = 0, 1, 3, 2, 6, 7, 5, 4`.

## Bài 2 — Điền chỗ trống (phản chiếu)

In [22]:
def exercise_2_gray_reflect(n: int) -> List[int]:
    result = [0]
    for bit in range(n):
        # TODO: thêm phần phản chiếu có bật bit `bit`
        pass
    return result


# assert exercise_2_gray_reflect(2) == [0, 1, 3, 2]
# assert exercise_2_gray_reflect(3) == [0, 1, 3, 2, 6, 7, 5, 4]

## Bài 3 — Biến thể: Circular Permutation in Binary Representation (LeetCode #1238)

Trả về một hoán vị của `0 .. 2^n − 1` bắt đầu bằng `start`, liền kề (kể cả cuối–đầu) khác một bit.

```python
circular_permutation(2, 3) -> [3, 2, 0, 1]
```

Gợi ý nhanh: XOR mọi phần tử của dãy Gray với `start` — XOR với một hằng số giữ nguyên tính "khác một bit".

In [23]:
def exercise_3_circular_permutation(n: int, start: int) -> List[int]:
    # TODO
    pass


# assert exercise_3_circular_permutation(2, 3) == [3, 2, 0, 1]
# r = exercise_3_circular_permutation(3, 2); assert r[0] == 2 and sorted(r) == list(range(8)) and all(bin(r[i] ^ r[(i + 1) % 8]).count("1") == 1 for i in range(8))

## Bài 4 — Biến thể: giải mã Gray (Gray → nhị phân)

Cho `g = i ^ (i >> 1)`, tìm lại `i`. Gợi ý nhanh: `i = g ^ (g >> 1) ^ (g >> 2) ^ …` (XOR mọi lần dịch cho tới khi bằng 0).

```python
gray_to_index(0b110) -> 4       # vì 4 ^ 2 = 6
```

In [24]:
def exercise_4_gray_to_index(g: int) -> int:
    # TODO
    pass


# assert exercise_4_gray_to_index(0b110) == 4
# assert all(exercise_4_gray_to_index(i ^ (i >> 1)) == i for i in range(1 << 12))

## Bài 5 — Mini challenge: Tháp Hà Nội và mã Gray

Ở bước thứ `i` (`i = 1 .. 2^n − 1`) của lời giải tối ưu Tháp Hà Nội `n` đĩa, đĩa được di chuyển là đĩa số `t + 1` với `t` = số bit 0 ở cuối của `i` (đĩa 1 nhỏ nhất). Đó cũng chính là bit bị lật giữa `g(i − 1)` và `g(i)`.

Viết hàm trả về danh sách số hiệu đĩa được di chuyển ở mỗi bước, và kiểm tra nó khớp với dãy bit bị lật của mã Gray.

```python
hanoi_disks(3) -> [1, 2, 1, 3, 1, 2, 1]
```

In [25]:
def exercise_5_hanoi_disks(n: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_hanoi_disks(3) == [1, 2, 1, 3, 1, 2, 1]
# assert exercise_5_hanoi_disks(1) == [1]
# assert all(exercise_5_hanoi_disks(n) == [((i ^ (i >> 1)) ^ ((i - 1) ^ ((i - 1) >> 1))).bit_length() for i in range(1, 1 << n)] for n in range(1, 10))

# 12. Gợi ý

**Bài 1**
- Mức 1: nửa sau = đảo ngược `G(3)`, cộng 8: `12, 13, 15, 14, 10, 11, 9, 8`.

**Bài 3**
- Mức 1: `[g ^ start for g in gray_formula(n)]`.

**Bài 4**
- Mức 1: `i = 0; while g: i ^= g; g >>= 1`.

**Bài 5**
- Mức 1: số bit 0 ở cuối của `i`: `(i & -i).bit_length() - 1`.
- Mức 2: `i & -i` giữ lại bit 1 thấp nhất của `i`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
g4 = gray_reflect(4)
print(g4)
print([f"{x:04b}" for x in g4])

[0, 1, 3, 2, 6, 7, 5, 4, 12, 13, 15, 14, 10, 11, 9, 8]
['0000', '0001', '0011', '0010', '0110', '0111', '0101', '0100', '1100', '1101', '1111', '1110', '1010', '1011', '1001', '1000']


## Đáp án Bài 2

In [27]:
def answer_2_gray_reflect(n: int) -> List[int]:
    result = [0]
    for bit in range(n):
        result += [x | (1 << bit) for x in reversed(result)]
    return result

assert answer_2_gray_reflect(2) == [0, 1, 3, 2]
assert answer_2_gray_reflect(3) == [0, 1, 3, 2, 6, 7, 5, 4]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [28]:
def answer_3_circular_permutation(n: int, start: int) -> List[int]:
    return [(i ^ (i >> 1)) ^ start for i in range(1 << n)]

assert answer_3_circular_permutation(2, 3) == [3, 2, 0, 1]
r = answer_3_circular_permutation(3, 2); assert r[0] == 2 and sorted(r) == list(range(8)) and all(bin(r[i] ^ r[(i + 1) % 8]).count("1") == 1 for i in range(8))
for n in range(1, 8):
    for start in range(1 << n):
        r = answer_3_circular_permutation(n, start)
        assert r[0] == start and sorted(r) == list(range(1 << n))
        assert all(one_bit_diff(r[i], r[(i + 1) % len(r)]) for i in range(len(r)))
print("Bài 3 PASS")

Bài 3 PASS

`(a ^ s) ^ (b ^ s) = a ^ b` — XOR cùng một số `s` không đổi tập bit khác nhau giữa hai số.

## Đáp án Bài 4

In [29]:
def answer_4_gray_to_index(g: int) -> int:
    i = 0
    while g:
        i ^= g
        g >>= 1
    return i

assert answer_4_gray_to_index(0b110) == 4
assert all(answer_4_gray_to_index(i ^ (i >> 1)) == i for i in range(1 << 12))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_hanoi_disks(n: int) -> List[int]:
    return [(i & -i).bit_length() for i in range(1, 1 << n)]

assert answer_5_hanoi_disks(3) == [1, 2, 1, 3, 1, 2, 1]
assert answer_5_hanoi_disks(1) == [1]
assert all(answer_5_hanoi_disks(n) == [((i ^ (i >> 1)) ^ ((i - 1) ^ ((i - 1) >> 1))).bit_length() for i in range(1, 1 << n)] for n in range(1, 10))

def hanoi_moves(n, src="A", dst="C", aux="B", out=None):
    out = [] if out is None else out
    if n:
        hanoi_moves(n - 1, src, aux, dst, out)
        out.append(n)
        hanoi_moves(n - 1, aux, dst, src, out)
    return out

assert all(hanoi_moves(n) == answer_5_hanoi_disks(n) for n in range(1, 12))
print("Bài 5 PASS (khớp với lời giải đệ quy Tháp Hà Nội)")

Bài 5 PASS (khớp với lời giải đệ quy Tháp Hà Nội)


# 14. Tổng kết

> Mã Gray: `g(i) = i ^ (i >> 1)` cho `i = 0 .. 2^n − 1`. Hoặc dựng bằng phản chiếu: `G(n) = G(n−1) + reversed(G(n−1)) với bit n−1 bật`. Cả hai `O(2^n)`.

---

## Checklist trước khi nộp

- [ ] Dùng `1 << n` (không phải `2 ^ n`)?
- [ ] Phản chiếu có `reversed`?
- [ ] Bắt đầu bằng `0`, đủ `2^n` số?
- [ ] Đầu và cuối cũng khác đúng một bit?